# Neural Networks from Scratch — 2. A Network from Scratch

**Notebook 2 of 4**

| # | Notebook | What you build |
|---|---|---|
| 1 | NumPy foundations | A single neuron, trained by hand |
| **2** | **A network from scratch** | **A 2-layer network with backpropagation** |
| 3 | The same network in PyTorch | Tensors, autograd, `nn.Module` |
| 4 | MNIST and a small CNN | A real image classifier |

---

## What this notebook is for

In notebook 1 you trained a single neuron and found its limit: it can only ever draw
a **straight line**. Here we break through that limit.

We will:

1. meet a problem that no straight line can solve
2. fix it by adding a **hidden layer**
3. work out **backpropagation** — the rule for training a network with more than one
   layer — and write it in about fifteen lines of NumPy
4. watch the network carve out a curved boundary, and then break it in instructive
   ways

This is the hardest of the four notebooks. It is also the one that matters most: once
backpropagation stops being mysterious, PyTorch is just convenience, and the rest of
deep learning is variations on what you'll have here.

Take it in two sittings if you like. Parts 4.1–4.3 are the theory; 4.4 onwards is
building and playing.

---
# Setup

This notebook stands alone — it re-defines everything it needs from notebook 1, so
you can run it fresh. The pieces should all look familiar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["figure.dpi"] = 110

rng = np.random.default_rng(seed=0)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def relu(z):
    return np.maximum(0.0, z)


def bce_loss(y_pred, y_true):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


def plot_decision_boundary(predict_fn, X, y, title="", ax=None):
    """Shade the plane by what predict_fn says, then scatter the real data on top."""
    if ax is None:
        _, ax = plt.subplots()
    pad = 0.5
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict_fn(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=50, cmap="coolwarm", alpha=0.7)
    ax.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=35)
    ax.set_title(title)
    ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2")
    return ax


print("ready")

---
# 4.1 A problem one neuron cannot solve

The classic example is **XOR**: four points, two classes. Output 1 when exactly one
input is 1, otherwise 0.

| x₁ | x₂ | label |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Look at the plot below and try to draw a single straight line with the two red points
on one side and the two blue points on the other. You can't. There is no such line —
this isn't a matter of not looking hard enough.

In [ ]:
X_xor = np.array([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]])
y_xor = np.array([0.0, 1.0, 1.0, 0.0])

plt.figure(figsize=(4.5, 4.5))
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm",
            edgecolor="k", s=300)
for (x1, x2), label in zip(X_xor, y_xor):
    plt.annotate(f"{int(label)}", (x1, x2), fontsize=13, fontweight="bold",
                 ha="center", va="center", color="white")
plt.xlim(-0.4, 1.4); plt.ylim(-0.4, 1.4)
plt.title("XOR - no straight line separates these")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()

XOR is a famous piece of history. In 1969 Minsky and Papert pointed out exactly this
limitation of single-layer networks, and interest in neural networks collapsed for
over a decade — the first "AI winter". The way out was known in principle (add more
layers) but nobody had a practical way to train the extra layers. That way is
backpropagation, popularised in 1986, and it is what we're about to build.

A more realistic version of the same problem is the **two moons** dataset: two
interleaving crescents. Again, no straight line will do — but a gentle curve
obviously would.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=400, noise=0.20, random_state=3)
y = y.astype(float)

print("X", X.shape, " y", y.shape)

plt.figure(figsize=(5.5, 4.5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=30)
plt.title("Two moons - needs a curve, not a line")
plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.show()

Let's confirm the single neuron fails, using exactly the code from notebook 1.

In [ ]:
def train_neuron(X, y, learning_rate=0.5, n_steps=500):
    n_samples, n_features = X.shape
    r = np.random.default_rng(0)
    w = r.normal(size=n_features) * 0.01
    b = 0.0
    for _ in range(n_steps):
        y_pred = sigmoid(X @ w + b)
        error = y_pred - y
        w -= learning_rate * (X.T @ error) / n_samples
        b -= learning_rate * np.mean(error)
    return w, b


w1n, b1n = train_neuron(X, y)
acc = np.mean((sigmoid(X @ w1n + b1n) > 0.5) == y)

plot_decision_boundary(lambda g: sigmoid(g @ w1n + b1n), X, y,
                       title=f"One neuron on two moons - accuracy {acc:.1%}")
plt.show()

It does its best — it finds the *best possible straight line* — and that best is
about 89%. Every point in the crooks of the two crescents is misclassified, and no
amount of extra training will help. The model is not big enough for the problem.

Keep that number in mind. **89% is the "straight line" score for this dataset**, and
for the rest of the notebook it's our benchmark: any model that scores around 89% has
not actually learned a curve, whatever its architecture claims.

The fix is to give it a hidden layer.

---
# 4.2 Adding a hidden layer

Here is the whole idea in one sentence: **instead of feeding the raw features
straight to the output neuron, first pass them through a layer of neurons whose job
is to invent better features.**

```
            input          hidden layer         output
          (2 features)     (8 neurons)        (1 neuron)

            x1  ----\      [ ]  ----\
                     >---- [ ]  -----> ( ) ----> probability
            x2  ----/      [ ]  ----/
                           ...
```

Each hidden neuron does exactly what you built in notebook 1: weighted sum, then
activation. So each one draws its own straight line across the input space and
reports which side a point fell on. The output neuron then combines those eight
straight-line answers.

And a combination of straight lines can be a curve. That's the trick — bend enough
lines together and you can fence off any region you like.

## The forward pass

With `h` hidden units, the whole network is four lines of NumPy:

```
Z1 = X  @ W1 + b1        (n, 2) @ (2, h) + (h,)  ->  (n, h)
A1 = relu(Z1)                                    ->  (n, h)
Z2 = A1 @ W2 + b2        (n, h) @ (h, 1) + (1,)  ->  (n, 1)
A2 = sigmoid(Z2)                                 ->  (n, 1)   the predictions
```

Track the shapes through that. Notice how `h` appears out of `W1`, travels through
the hidden layer, and is absorbed by `W2`. The input width (2) and output width (1)
are fixed by the problem; `h` is ours to choose.

## Initialising the parameters

Two rules, both of which matter more than they look:

**Don't start at zero.** If every hidden neuron starts with identical weights, they
all compute the same thing, they all receive the same gradient, and they stay
identical forever. Your 8-neuron layer behaves like 1 neuron. This is the *symmetry
problem*, and random initialisation is what breaks it. (We'll demonstrate it in 4.6.)

**Don't start too big either.** Scale the random numbers by `sqrt(2 / n_inputs)` —
this is *He initialisation*, the standard choice for ReLU layers. It keeps the signal
from growing or shrinking as it passes through the network. You don't need to
remember the formula, just that sensible default scaling exists and frameworks apply
it for you.

In [ ]:
def init_params(n_input, n_hidden, seed=0):
    """Create the four parameter arrays. Returns a dict so we can pass them around."""
    r = np.random.default_rng(seed)
    return {
        "W1": r.normal(size=(n_input, n_hidden)) * np.sqrt(2.0 / n_input),
        "b1": np.zeros(n_hidden),
        "W2": r.normal(size=(n_hidden, 1)) * np.sqrt(2.0 / n_hidden),
        "b2": np.zeros(1),
    }


def forward(X, p):
    """Run the network. Returns predictions plus the intermediate values.

    We keep Z1, A1, Z2 around because the backward pass needs them - that is
    the real reason frameworks talk about 'storing activations' and why
    training uses so much more memory than prediction does.
    """
    Z1 = X @ p["W1"] + p["b1"]      # (n, h)
    A1 = relu(Z1)                   # (n, h)
    Z2 = A1 @ p["W2"] + p["b2"]     # (n, 1)
    A2 = sigmoid(Z2)                # (n, 1)
    return A2, {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}


params = init_params(n_input=2, n_hidden=8)
for name, arr in params.items():
    print(f"{name}: shape {str(arr.shape):8}")

y_hat, cache = forward(X, params)
print()
print("predictions shape:", y_hat.shape)
print("first 3 (untrained, so near 0.5):", np.round(y_hat[:3, 0], 3))

> **A shape detail worth noting.** The predictions come out as `(n, 1)` — a column —
> while our labels `y` are `(n,)` — flat. If we subtract them directly, broadcasting
> will silently produce an `(n, n)` array and everything downstream will be nonsense
> without raising a single error. This is the single most common bug in from-scratch
> network code.
>
> The fix is to reshape the labels to `(n, 1)` once, up front, and keep everything in
> column form. We'll do that in the training loop.

In [ ]:
# See the trap for yourself - this is why we reshape.
print("y_hat", y_hat.shape, "- y", y.shape, "->", (y_hat - y).shape, " <- WRONG")
print("y_hat", y_hat.shape, "- y.reshape(-1,1)", y.reshape(-1, 1).shape,
      "->", (y_hat - y.reshape(-1, 1)).shape, " <- right")

---
# 4.3 Backpropagation

Now the part everyone finds intimidating. It isn't, really — it's **blame
assignment**, applied backwards through the network.

### The intuition

The output was wrong by some amount. Who is responsible?

- The **output neuron's weights** are directly responsible: each one scaled some
  hidden unit's signal. A weight attached to a hidden unit that was firing strongly
  gets a lot of the blame; a weight attached to a silent unit gets none.
- The **hidden units** are indirectly responsible. A hidden unit's share of the blame
  is the output error *multiplied by the weight connecting it to the output*. A unit
  connected with a big weight had a big influence, so it gets a big share.
- Once a hidden unit knows its share, it does exactly the same thing again for *its*
  inputs.

So the error flows backwards, getting split up at each layer according to the weights
it travelled along. That's it. The chain rule is the formal statement of "multiply
the blame as it passes back through each step".

### One more wrinkle: the ReLU gate

ReLU outputs zero for any negative input. If a hidden unit was *off* for a given
example, it contributed nothing to the output, so it deserves none of the blame for
that example. In code that's a multiplication by `(Z1 > 0)` — a mask of ones and
zeros that lets blame through the units that were on and blocks it at the ones that
were off.

### The equations

Here they are, with the shape of each next to it. `n` is the number of examples.

```
dZ2 = (A2 - Y) / n              (n, 1)   output error, averaged over the batch
dW2 = A1.T @ dZ2                (h, 1)   blame x the signal each weight carried
db2 = dZ2.sum(axis=0)           (1,)

dA1 = dZ2 @ W2.T                (n, h)   blame flowing back to the hidden units
dZ1 = dA1 * (Z1 > 0)            (n, h)   blocked where ReLU was off
dW1 = X.T @ dZ1                 (2, h)
db1 = dZ1.sum(axis=0)           (h,)
```

Read it top to bottom and notice the rhythm: **error, then weights, then pass it
back, then repeat.** Every layer of every network is trained by those same four
moves, and a 200-layer network is this loop 200 times.

Compare `dW2 = A1.T @ dZ2` with the `grad_w = X.T @ error / n` from notebook 1. It's
the same formula — the hidden activations `A1` are simply playing the role that the
raw inputs `X` played before. That is the sense in which the hidden layer "invents
features": to the output neuron, they *are* the features.

In [ ]:
def backward(X, Y, p, cache):
    """Compute the gradient of the loss w.r.t. every parameter.

    X: (n, 2)   Y: (n, 1)   -> returns a dict shaped like p
    """
    n = X.shape[0]
    A1, Z1, A2 = cache["A1"], cache["Z1"], cache["A2"]

    # ---- output layer ----
    dZ2 = (A2 - Y) / n                  # (n, 1)
    dW2 = A1.T @ dZ2                    # (h, 1)
    db2 = dZ2.sum(axis=0)               # (1,)

    # ---- pass the blame back to the hidden layer ----
    dA1 = dZ2 @ p["W2"].T               # (n, h)
    dZ1 = dA1 * (Z1 > 0)                # (n, h)  ReLU gate

    # ---- hidden layer ----
    dW1 = X.T @ dZ1                     # (2, h)
    db1 = dZ1.sum(axis=0)               # (h,)

    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


grads = backward(X, y.reshape(-1, 1), params, cache)
for name in params:
    print(f"{name}: param {str(params[name].shape):8} gradient {str(grads[name].shape):8}"
          f" {'MATCH' if params[name].shape == grads[name].shape else 'MISMATCH'}")

**Every gradient has the same shape as the parameter it belongs to.** That must be
true — the gradient tells each individual number which way to move, so there has to
be exactly one gradient per parameter. It's a free correctness check, and worth
asserting in your own code.

## 4.3.1 Checking the gradients numerically

Hand-derived gradients are easy to get subtly wrong, and a subtly wrong gradient
doesn't crash — it just trains badly, which is much harder to notice. So here's the
standard safety net.

The definition of a gradient is "how much does the loss change if I nudge this
parameter a tiny bit?" We can measure that directly: change one number by `1e-5`,
see how much the loss moves, divide. Slow (one forward pass per parameter) but
assumption-free. If our fast analytic gradients match the slow measured ones, the
maths is right.

This technique is called **gradient checking**, and it's how people debugged networks
before autograd existed. If you ever write a custom layer, you'll reach for it again.

In [ ]:
def numerical_gradient(X, Y, p, name, index, eps=1e-6):
    """Measure one gradient by brute force: nudge the parameter, see what the loss does."""
    p_plus = {k: v.copy() for k, v in p.items()}
    p_minus = {k: v.copy() for k, v in p.items()}
    p_plus[name][index] += eps
    p_minus[name][index] -= eps

    loss_plus = bce_loss(forward(X, p_plus)[0], Y)
    loss_minus = bce_loss(forward(X, p_minus)[0], Y)
    return (loss_plus - loss_minus) / (2 * eps)


Y = y.reshape(-1, 1)
print(f"{'parameter':12} {'analytic':>14} {'numerical':>14} {'difference':>12}")
print("-" * 55)
for name, index in [("W1", (0, 0)), ("W1", (1, 3)), ("b1", (2,)),
                    ("W2", (0, 0)), ("W2", (5, 0)), ("b2", (0,))]:
    analytic = grads[name][index]
    numeric = numerical_gradient(X, Y, params, name, index)
    print(f"{name}{str(index):10} {analytic:14.9f} {numeric:14.9f} {abs(analytic-numeric):12.2e}")

Agreement to about nine decimal places. The backward pass is correct.

(The small leftover difference is floating-point rounding in the numerical estimate,
not an error in our maths. A mismatch in the *third* decimal place would be the
warning sign.)

We will do this exact comparison once more in notebook 3 — against PyTorch's autograd
instead of against finite differences.

---
# 4.4 The training loop

Forward, loss, backward, update. The same five steps as notebook 1 — only the model
in the middle got bigger.

In [ ]:
def train_network(X, y, n_hidden=8, learning_rate=0.5, n_steps=3000,
                  seed=0, snapshot_at=()):
    """Train the 2-layer network. Returns params, loss history, and any snapshots."""
    Y = y.reshape(-1, 1)                       # (n, 1) - column form, always
    p = init_params(X.shape[1], n_hidden, seed=seed)

    losses = []
    snapshots = {}
    for step in range(n_steps):
        if step in snapshot_at:
            snapshots[step] = {k: v.copy() for k, v in p.items()}

        y_hat, cache = forward(X, p)           # 1. forward
        losses.append(bce_loss(y_hat, Y))      # 2. loss
        grads = backward(X, Y, p, cache)       # 3. backward

        for k in p:                            # 4. update
            p[k] -= learning_rate * grads[k]

    if n_steps in snapshot_at:
        snapshots[n_steps] = {k: v.copy() for k, v in p.items()}
    return p, np.array(losses), snapshots


trained, losses, _ = train_network(X, y, n_hidden=8, learning_rate=0.5, n_steps=3000)

def predict_proba(grid, p):
    return forward(grid, p)[0].ravel()         # ravel: (n,1) -> (n,) for plotting

acc = np.mean((predict_proba(X, trained) > 0.5) == y)
print(f"loss  {losses[0]:.4f}  ->  {losses[-1]:.4f}")
print(f"accuracy: {acc:.1%}   (the single neuron managed about 89%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(losses, linewidth=2)
axes[0].set_xlabel("training step"); axes[0].set_ylabel("loss")
axes[0].set_title("Loss")

plot_decision_boundary(lambda g: predict_proba(g, trained), X, y,
                       title=f"2-layer network - {acc:.1%}", ax=axes[1])
plt.tight_layout()
plt.show()

A curve. Eight straight lines, bent together by the output neuron into a boundary
that follows the shape of the data.

Look closely at the boundary and you can see it is made of **straight segments** with
kinks between them — that's ReLU showing through. With more hidden units the segments
get shorter and the boundary looks smoother, but it is always piecewise linear
underneath.

## Watching it learn

In [ ]:
steps = [0, 50, 200, 1000, 3000]
_, _, snaps = train_network(X, y, n_hidden=8, learning_rate=0.5,
                            n_steps=3000, snapshot_at=steps)

fig, axes = plt.subplots(1, len(steps), figsize=(4 * len(steps), 4))
for ax, s in zip(axes, steps):
    p_s = snaps[s]
    a = np.mean((predict_proba(X, p_s) > 0.5) == y)
    plot_decision_boundary(lambda g, pp=p_s: predict_proba(g, pp), X, y,
                           title=f"step {s} - {a:.0%}", ax=ax)
plt.tight_layout()
plt.show()

Step 0 is the random initialisation — a vague, almost-straight split. By step 200 it
has found the broad diagonal. Between 200 and 3000 it works on the curve, bending
into the crooks of the two crescents.

That progression — get the rough answer fast, then spend most of training on the
details — is what almost all neural network training looks like.

---
# 4.5 Back to XOR

Four points, two hidden units. The smallest interesting neural network there is.

In [ ]:
p_xor, losses_xor, _ = train_network(X_xor, y_xor, n_hidden=4,
                                     learning_rate=1.0, n_steps=5000, seed=1)

pred = predict_proba(X_xor, p_xor)
print("  x1   x2   target   predicted")
for (x1, x2), t, q in zip(X_xor, y_xor, pred):
    print(f"  {x1:.0f}    {x2:.0f}      {t:.0f}        {q:.3f}")

plot_decision_boundary(lambda g: predict_proba(g, p_xor), X_xor, y_xor,
                       title="XOR, solved")
plt.show()

Solved — and not by memorising four points, but by carving the plane into regions
that happen to put the right points together. The thing that stalled the field for
fifteen years takes a modern laptop a fraction of a second.

**🔬 Try it.** Set `n_hidden=1` and re-run. It fails, and it must: one hidden unit is
one straight line, and we're right back where we started. Then try `n_hidden=2`,
which is the theoretical minimum — it works sometimes, depending on the random `seed`.
Try a few seeds. That sensitivity is real, and it is one reason people use more
hidden units than the bare minimum.

---
# 4.6 🔬 Four experiments

The best way to understand a mechanism is to break it. Each of these breaks one thing
and shows you the consequence.

## Experiment 1: remove the activation function

This is the important one. We claimed in notebook 1 that without a non-linear
activation, depth buys you nothing. Let's prove it by replacing ReLU with the
identity function — the hidden layer still exists, still has eight units, still
trains, but no longer bends anything.

In [ ]:
def forward_linear(X, p):
    """Identical to forward(), but the hidden activation is removed."""
    Z1 = X @ p["W1"] + p["b1"]
    A1 = Z1                            # <-- the only change: no relu
    Z2 = A1 @ p["W2"] + p["b2"]
    A2 = sigmoid(Z2)
    return A2, {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}


def backward_linear(X, Y, p, cache):
    n = X.shape[0]
    dZ2 = (cache["A2"] - Y) / n
    dW2 = cache["A1"].T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ p["W2"].T
    dZ1 = dA1                          # <-- no ReLU gate either
    return {"W1": X.T @ dZ1, "b1": dZ1.sum(axis=0), "W2": dW2, "b2": db2}


p_lin = init_params(2, 8, seed=0)
Y = y.reshape(-1, 1)
for _ in range(3000):
    yh, c = forward_linear(X, p_lin)
    p_lin_grads = backward_linear(X, Y, p_lin, c)
    for k in p_lin:
        p_lin[k] -= 0.5 * p_lin_grads[k]

acc_lin = np.mean((forward_linear(X, p_lin)[0].ravel() > 0.5) == y)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_decision_boundary(lambda g: forward_linear(g, p_lin)[0].ravel(), X, y,
                       title=f"8 hidden units, NO activation - {acc_lin:.1%}", ax=axes[0])
plot_decision_boundary(lambda g: predict_proba(g, trained), X, y,
                       title=f"8 hidden units, WITH ReLU - {acc:.1%}", ax=axes[1])
plt.tight_layout()
plt.show()

A perfectly straight line, scoring the same ~89% as the single neuron. Same
architecture, same number of parameters, same training — and it has collapsed back
into notebook 1.

The reason is algebra: `(X @ W1 + b1) @ W2 + b2` can be multiplied out into
`X @ (W1 @ W2) + (b1 @ W2 + b2)`, which is just `X @ W + b` for some other `W` and
`b`. Two linear layers *are* one linear layer. Stack a thousand and it is still one.

**The activation function is not a detail. It is the thing that makes depth mean
anything at all.**

## Experiment 2: how many hidden units?

Each hidden unit contributes one straight line for the output neuron to work with.
More units means more lines, so a more flexible boundary. Let's sweep the size and
see where the curve appears.

Remember the benchmark: **89% means "it found a straight line"**.

In [ ]:
sizes = [1, 2, 4, 8, 32, 128]
fig, axes = plt.subplots(1, len(sizes), figsize=(3.6 * len(sizes), 3.8))

for ax, h in zip(axes, sizes):
    p_h, _, _ = train_network(X, y, n_hidden=h, learning_rate=0.5, n_steps=4000)
    a = np.mean((predict_proba(X, p_h) > 0.5) == y)
    plot_decision_boundary(lambda g, pp=p_h: predict_proba(g, pp), X, y,
                           title=f"{h} hidden units - {a:.0%}", ax=ax)

plt.tight_layout()
plt.show()

This may not be the tidy progression you expected, and that is worth dwelling on.

**1, 2 and 4 units are all stuck at roughly the straight-line score.** Not "a slightly
worse curve" — no curve at all. Then somewhere around 8 units the model suddenly
finds the shape, and after that extra units change very little.

So the jump isn't gradual. Why not?

## 2b: small networks are unreliable

The theory says two hidden units are *enough* to bend a curve. The practice is that
training doesn't reliably find that solution. Gradient descent goes downhill from
wherever it happens to start, and with very few units there are plenty of mediocre
valleys to get stuck in. Whether you land in a good one is largely luck — which
means it depends on the random seed.

Let's measure that directly.

In [ ]:
print("accuracy by hidden size and random seed")
print(f"{'h':>4} | " + "  ".join(f"seed {s}" for s in range(6)))
print("-" * 52)
for h in [2, 3, 4, 8, 32]:
    row = []
    for s in range(6):
        p_h, _, _ = train_network(X, y, n_hidden=h, learning_rate=0.5,
                                  n_steps=4000, seed=s)
        row.append(np.mean((predict_proba(X, p_h) > 0.5) == y))
    print(f"{h:>4} | " + "  ".join(f"{v:5.1%} " for v in row))

Look at the rows for `h=3` and `h=4`: the same code, the same data, the same number
of steps — and the result flips between "found the curve" (~97%) and "found a line"
(~89%) depending on nothing but the starting random numbers. By `h=8` the row is
consistent, and by `h=32` it is boringly reliable.

**This is why real networks are built much wider than the theoretical minimum.** The
extra units aren't there because the problem needs them; they're there so that
training reliably *finds* a good solution. Redundancy buys robustness.

### Dead ReLU units

There's a specific mechanism behind some of those failures, and it has a name.

If a ReLU unit's weights drift to where it outputs zero for every training example,
then it contributes nothing to the output — and, because of the `(Z1 > 0)` gate we
wrote in the backward pass, it also receives *zero gradient*. It can never recover.
The unit is **dead**, permanently.

In [ ]:
for h in [4, 32]:
    p_h, _, _ = train_network(X, y, n_hidden=h, learning_rate=0.5, n_steps=4000)
    Z1 = X @ p_h["W1"] + p_h["b1"]
    active_fraction = (Z1 > 0).mean(axis=0)       # per unit: how often is it on?
    dead = (active_fraction == 0).sum()
    print(f"h={h:3}:  {dead} dead units out of {h}")
    if h == 4:
        print("        fraction of examples each unit fires on:",
              np.round(active_fraction, 3))

A dead unit in a layer of 4 is a quarter of your model gone — and that is exactly the
run that scored ~89%. The wide layer has none dead at all here, and even if one or
two did die it would hardly notice. Same "redundancy buys robustness" lesson, from a
different angle.

(Fixes exist: `LeakyReLU` leaks a small gradient through for negative inputs, and a
lower learning rate makes units less likely to get knocked out in the first place.
Both are one-liners in PyTorch.)

### And the very large networks?

The 128-unit model scores about the same as the 32-unit one, but look at its
boundary: more intricate, with small excursions towards individual points near the
edges. Is that a better model, or is it starting to memorise noise?

**We cannot tell from this plot**, because we've been scoring accuracy on the very
data we trained on — which is marking your own homework. A model that memorised all
400 points perfectly would score 100% here and be useless on anything new.

Fixing that requires holding data back, which is what notebook 4 does.

## Experiment 3: initialising at zero

Earlier we claimed that starting all weights at zero breaks the network, because
every hidden unit stays identical to every other. Let's check.

In [ ]:
p_zero = {"W1": np.zeros((2, 8)), "b1": np.zeros(8),
          "W2": np.zeros((8, 1)), "b2": np.zeros(1)}

for _ in range(3000):
    yh, c = forward(X, p_zero)
    g_ = backward(X, Y, p_zero, c)
    for k in p_zero:
        p_zero[k] -= 0.5 * g_[k]

acc_zero = np.mean((predict_proba(X, p_zero) > 0.5) == y)
print(f"accuracy with all-zero init: {acc_zero:.1%}")
print()
print("W1 after training (one column per hidden unit):")
print(np.round(p_zero["W1"], 4))

Every column is identical — all eight hidden units learned exactly the same thing, so
the layer has the capacity of a single unit. (With ReLU and zero weights it's worse
still: the units start switched off, and nothing can turn them back on.)

Random initialisation isn't a superstition or a nicety. It is what makes the units
*different from each other*, and different is what lets them specialise. Every
framework does this for you — but when you see `torch.nn.Linear` in notebook 3
quietly filling itself with random numbers, you'll know why.

## Experiment 4: learning rate, again

Same lesson as notebook 1, now with more parameters to go wrong at once.

In [ ]:
plt.figure(figsize=(7.5, 4.5))
for lr in [0.01, 0.1, 1.0, 5.0, 50.0]:
    _, l, _ = train_network(X, y, n_hidden=8, learning_rate=lr, n_steps=2000)
    plt.plot(l, label=f"lr = {lr}", linewidth=2)

plt.xlabel("training step"); plt.ylabel("loss")
plt.title("Learning rate on the 2-layer network")
plt.legend()
plt.show()

Read the curves as diagnoses you'll use on your own models:

- **flat and high** → learning rate too small (or the model is too simple)
- **smooth downward slope** → healthy
- **jagged, bouncing** → too large; it is stepping over the minimum
- **shoots up, or goes NaN** → far too large

That vocabulary is most of practical model debugging. When someone says "the loss
curve looks wrong", this is what they're reading.

---
# Recap

You have written a neural network from nothing. Specifically:

- **Why depth needs non-linearity** — you proved it by deleting the activation and
  watching the network collapse to a straight line
- **The forward pass** — four lines: matmul, activation, matmul, activation
- **Backpropagation** — blame flowing backwards, split by the weights it travels
  along and gated by ReLU
- **Gradient checking** — how to prove a hand-derived gradient is right
- **Initialisation** — why random, and why scaled
- **Reading loss curves** — flat, smooth, jagged, exploding

That is the complete mechanism. Everything in modern deep learning — transformers,
diffusion models, all of it — is this same loop with different arrangements of layers
in the middle.

### Check yourself

1. Why can't two `Linear` layers with nothing between them do more than one?
2. `dW2 = A1.T @ dZ2`. Why `A1` and not `X`?
3. What would happen if you initialised `W1` randomly but `W2` to all zeros?
   (Try it — the answer is more interesting than you might expect.)
4. Your loss curve is jagged and bouncing around. What do you change first?
5. Your network gets 100% on training data and 70% on data it hasn't seen. What is
   happening, and would more hidden units help or hurt?

### Optional stretch

- Add a **second** hidden layer. You'll need one more matmul in `forward` and one
  more blame-passing step in `backward`. It is fiddly, and that fiddliness is
  precisely what PyTorch removes — which makes it excellent preparation for
  notebook 3.
- Swap ReLU for sigmoid in the hidden layer. The gradient becomes
  `dZ1 = dA1 * A1 * (1 - A1)`. Compare training speed — you should find it slower,
  which is much of why ReLU won.
- Try `make_circles` from `sklearn.datasets` instead of `make_moons`.

### Next

**Notebook 3 — The same network in PyTorch.** We build this exact network again, and
PyTorch computes the backward pass for us. We'll check its gradients against the ones
we derived here, and find they agree to the last decimal place.